# Eligibility File QA — Healthcare Data Operations

**Objective:** Validate a member eligibility file against business rules,
identify data quality issues, and assess downstream impact before
a claims processing run.

**Why this matters:** In healthcare data ops, a single eligibility error
can result in a denied claim, a compliance violation, or a patient
being incorrectly billed. This is not "just data cleaning" — it's
patient safety work.

**Dataset:** 228 synthetic member eligibility records with
deliberate errors introduced to simulate real-world messiness.

In [ ]:
# Setup
import pandas as pd
import numpy as np
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')

# Config
TODAY = date(2026, 3, 25)
CHILD_MAX_AGE = 26
VALID_PLANS = {'HMO', 'PPO', 'EPO', 'HDHP', 'POS'}
VALID_METAL_LEVELS = {'Bronze', 'Silver', 'Gold', 'Platinum'}

print(f"Analysis date: {TODAY}")
print(f"Child max age: {CHILD_MAX_AGE}")

## Step 1 — Load the File

In [ ]:
# Load eligibility file
df = pd.read_csv('data/eligibility_dirty.csv')

print(f"Total records: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
df.head()

## Step 2 — Initial Profile

Get a quick picture of the data before diving into validation.

In [ ]:
# Shape check
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

# Null / empty value summary
print("\nEmpty string counts:")
for col in df.columns:
    empty = (df[col] == '').sum()
    null = df[col].isnull().sum()
    if empty > 0 or null > 0:
        print(f"  {col}: empty={empty}, null={null}")

# Sample of each column
df.describe(include='all').T

## Step 3 — Zip Code Validation

**Issue:** Zip codes stored as integers cause Northeast zips (001xx) to
lose their leading zeros. 02134 becomes 2134. This breaks geocoding,
address matching, and CMS reporting.

**Downstream impact:** Claims from MA, NY, CT members will have
invalid zip codes in the system. Any geocoding or regional reporting
will be incorrect for these members.

In [ ]:
# Detect zip codes stored as integers (losing leading zeros)
df['zip_code_int'] = pd.to_numeric(df['zip_code'], errors='coerce')

# Valid zip = 5 digits
df['zip_has_leading_zero_issue'] = (
    (df['zip_code_int'] < 10000) & (df['zip_code_int'] > 0)
).astype(int)

zip_issues = df[df['zip_has_leading_zero_issue'] == 1]
print(f"⚠️  Zip codes with leading-zero issue: {len(zip_issues)}")
print(f"   States affected: {zip_issues['state'].value_counts().to_dict()}")
print(f"\nExamples of truncated zips:")
print(zip_issues[['mem_id', 'state', 'zip_code', 'zip_code_int']].head(10).to_string())

print(f"\n📋 DOWNSTREAM IMPACT: {len(zip_issues)} members will have")
print(f"   incorrect zip codes in claims processing.")
print(f"   Fix: Cast zip_code to string and zero-pad to 5 digits.")

## Step 4 — Duplicate Member ID Detection

**Issue:** Duplicate `mem_id` values mean multiple records point to
the same member. This is a hard blocker — claims cannot be reliably
adjudicated when member identity is ambiguous.

**Root causes in real-world data:** System mergers, open enrollment
data entry errors, fraud.

**Downstream impact:** Claims will be misrouted or denied. Duplicate
members inflate per-capita cost calculations.

In [ ]:
# Find duplicate mem_ids
dup_mask = df['mem_id'].duplicated(keep=False)
duplicates = df[dup_mask].sort_values('mem_id')

print(f"🚨 DUPLICATE mem_ids found: {duplicates['mem_id'].nunique()} unique IDs")
print(f"   Affects {len(duplicates)} total records")
print(f"\nDuplicate records:")
print(duplicates[['mem_id', 'first_name', 'last_name', 'dob', 'email', 'effective_date', 'termination_date']].to_string())

print(f"\n📋 DOWNSTREAM IMPACT: Claims for these members cannot be reliably")
print(f"   processed until deduplication is resolved.")
print(f"   Fix: Investigate each pair — one record may be correct,")
print(f"   the other should be flagged as a duplicate and excluded.")

## Step 5 — Age Consistency Check (Covered Relation vs DOB)

**Issue:** Members marked as "Child" with ages above 26 are likely
data entry errors. Dependents lose coverage at 26 in most plans.

**Downstream impact:** Claims for these "children" will be denied
post-26. If they're being billed as dependents, the adult members
are overpaying for coverage that shouldn't exist.

In [ ]:
# Calculate age as of today
def calc_age(dob_str):
    try:
        dob = datetime.strptime(str(dob_str), '%Y-%m-%d').date()
        age = (TODAY - dob).days / 365.25
        return round(age, 1)
    except:
        return None

df['age'] = df['dob'].apply(calc_age)

# Flag children over 26
child_overage = df[
    (df['covered_relation'] == 'Child') & 
    (df['age'].notna()) & 
    (df['age'] > CHILD_MAX_AGE)
]

print(f"⚠️  Dependents (Child) over age {CHILD_MAX_AGE}: {len(child_overage)}")
print(f"\nAge distribution for 'Child' records:")
print(df[df['covered_relation'] == 'Child']['age'].describe())

print(f"\nRecords flagged:")
print(child_overage[['mem_id', 'first_name', 'age', 'covered_relation', 'dob']].head(10).to_string())

print(f"\n📋 DOWNSTREAM IMPACT: {len(child_overage)} members may lose")
print(f"   dependent coverage incorrectly, or employer is paying for")
print(f"   ineligible dependents.")
print(f"   Fix: Verify with HR/benefits admin — likely DOB error or")
print(f"   covered_relation field needs correction.")

## Step 6 — Missing Required Fields

**Issue:** Missing DOB, email, or phone are soft blockers depending on
business rules. Email/phone are contact channels — missing is bad.
DOB is critical for actuarial calculations and identity verification.

**Downstream impact:** Members with missing DOB cannot be accurately
risk-adjusted. Missing contact info blocks outreach, billing reminders.

In [ ]:
required = ['dob', 'email', 'phone']

print("Missing required fields:")
for field in required:
    missing = df[df[field] == '']
    print(f"  {field}: {len(missing)}")

# Show missing records
missing_dob = df[df['dob'] == '']
if len(missing_dob) > 0:
    print(f"\nRecords missing DOB:")
    print(missing_dob[['mem_id', 'first_name', 'last_name', 'email', 'phone']].to_string())

print(f"\n📋 DOWNSTREAM IMPACT: Missing DOB prevents risk scoring and")
print(f"   may block enrollment. Missing contact info prevents billing")
print(f"   outreach.")
print(f"   Fix: Initiate data capture workflow with member.")

## Step 7 — Active-Only Filter

**Issue:** A small number of records have termination dates in the past
but are still marked as "Active" in the system. These should be
excluded from current active roster runs.

**Downstream impact:** Active-only rosters passed to claims processing
will include terminated members, causing post-termination claims to
error or be incorrectly adjudicated.

In [ ]:
# Find terminated members incorrectly marked as active
def parse_termination(term_str):
    if term_str == 'Active':
        return None
    try:
        return datetime.strptime(str(term_str), '%Y-%m-%d').date()
    except:
        return None

df['termination_date_parsed'] = df['termination_date'].apply(parse_termination)

# Active records with past termination dates
inactive_active = df[
    (df['termination_date'] == 'Active') &
    (df['termination_date_parsed'].notna()) &
    (df['termination_date_parsed'] < TODAY)
]

print(f"⚠️  Terminated members incorrectly marked as 'Active': {len(inactive_active)}")
if len(inactive_active) > 0:
    print(inactive_active[['mem_id', 'first_name', 'last_name', 'termination_date']].to_string())

print(f"\n📋 DOWNSTREAM IMPACT: These members should be excluded from")
print(f"   current active roster. Claims submitted for them will fail.")
print(f"   Fix: Apply termination_date filter before roster export.")

## Step 8 — Produce Clean Roster

Apply all fixes and produce the clean active roster.

In [ ]:
# Build clean roster
clean = df.copy()

# Fix 1: Zero-pad zip codes
clean['zip_code'] = clean['zip_code_int'].apply(
    lambda x: str(int(x)).zfill(5) if pd.notna(x) else ''
)

# Fix 2: Remove duplicates (keep first occurrence)
clean = clean.drop_duplicates(subset='mem_id', keep='first')

# Fix 3: Exclude inactive
clean = clean[
    (clean['termination_date'] == 'Active') | 
    (clean['termination_date_parsed'].isna()) | 
    (clean['termination_date_parsed'] >= TODAY)
]

# Fix 4: Remove records with missing DOB
clean = clean[clean['dob'] != '']

print(f"Roster transformation:")
print(f"  Original:  {len(df)} records")
print(f"  After dedup: {len(df.drop_duplicates(subset='mem_id', keep='first'))} records")
print(f"  After active filter: {len(df[(df['termination_date'] == 'Active') | (df['termination_date_parsed'].isna()) | (df['termination_date_parsed'] >= TODAY)])} records")
print(f"  After DOB filter: {len(clean)} records")
clean.to_csv('../data/eligibility_clean.csv', index=False)
print(f"\n✅ Clean roster saved: data/eligibility_clean.csv ({len(clean)} records)")

## Step 9 — QA Summary

Aggregate findings and document downstream impact.

In [ ]:
print("=" * 60)
print("ELIGIBILITY QA — FINAL SUMMARY")
print("=" * 60)

issues = []

if len(zip_issues) > 0:
    issues.append(f"  🚨 ZIP LEADING ZEROS: {len(zip_issues)} records — zip codes truncated")
    issues.append(f"     Fix: Zero-pad to 5 digits. Re-geocode affected addresses.")
    issues.append(f"     Impact: Claims geocoding and regional reporting affected.")

if len(duplicates) > 0:
    issues.append(f"  🚨 DUPLICATE MEM_IDs: {len(duplicates)} records ({duplicates['mem_id'].nunique()} IDs)")
    issues.append(f"     Fix: Deduplicate — keep correct record, archive other.")
    issues.append(f"     Impact: Claims cannot be reliably processed for these members.")

if len(child_overage) > 0:
    issues.append(f"  ⚠️  DEPENDENT AGE MISMATCH: {len(child_overage)} records")
    issues.append(f"     Fix: Verify DOB and covered_relation with HR.")
    issues.append(f"     Impact: Incorrect dependent billing, post-26 claim denials.")

if (df['dob'] == '').sum() > 0:
    issues.append(f"  ⚠️  MISSING DOB: {(df['dob'] == '').sum()} records")
    issues.append(f"     Fix: Initiate data capture workflow.")
    issues.append(f"     Impact: Cannot perform risk scoring.")

if len(inactive_active) > 0:
    issues.append(f"  ⚠️  TERMINATION DATE INCONSISTENCY: {len(inactive_active)} records")
    issues.append(f"     Fix: Apply termination filter before roster export.")
    issues.append(f"     Impact: Claims for terminated members will fail.")

if not issues:
    print("✅ No issues found.")
else:
    for issue in issues:
        print(issue)

print(f"\n{'=' * 60}")
print(f"NET RESULT: {len(df)} input → {len(clean)} clean records")
print(f"Error rate: {round((1 - len(clean)/len(df)) * 100, 1)}%")
print(f"{'=' * 60}")